# Bag-of-Words MaxRL (word-heteroskedastic, 128 rollouts)

Zero-step-sampling MaxRL against the canonical signal-heteroskedastic dataset: per-prompt noise std varies with prompt composition while the global `Corr(signal, target) = corr` is preserved.

The policy is a Gaussian around the model's scalar prediction, m_theta(z | x) = Normal(f_theta(x), sigma^2). MaxRL weights use the Gaussian likelihood of the noisy target under each rollout.

In [1]:
import polars as pl
import torch

from src import get_repo_base
from src.experiments.bag_of_words.maxrl import BagOfWordsMaxRLConfig

from src.experiments.bag_of_words.analysis import (
    BagOfWordsAnalysisConfig,
    corr_expr,
    rsq_expr,
)

repo_root = get_repo_base()
device = torch.device("cuda:0")

/home/nlyu/Code/maxrl-statistics/src/experiments/bag_of_words/maxrl.py:12: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


## Configure

In [2]:
config = BagOfWordsMaxRLConfig.get_canonical(
    dataset="word_heteroskedastic",
    dataset_base_folder=repo_root / "artifacts" / "bow-data",
    study_base_folder=repo_root / "artifacts" / "bow-maxrl-word-het-example-r128",
    corr=0.2,
    aux_words_ratio=0.5,
    train_epochs=2,
    num_rollouts_per_sample=128,
    gaussian_stdev=1.0,
    subtract_baseline=True,
)

display(config.visualize())
print(f"Study folder: {config.study_folder}")
print(f"Dataset corr target: {config.data.corr:.4f}")
print(f"SNR halflife (word quantile): {config.data.snr_halflife_in_word_quantile}")
print(f"Backbone lr: {config.optimizer.lr:.3e}")
print(f"Head lr:     {config.optimizer.head_lr:.3e}")
print(f"Rollouts/sample: {config.num_rollouts_per_sample}")
print(f"Canonical degree: {config.degree}")
print(f"Policy stdev:     {config.gaussian_stdev}")

state = config.initialize(device=device)
display(
    state.dataset.token_lengths_plot(
        filter_threshold=config.tokenization.filter_samples_above_n_tokens,
    )
)

Study folder: /home/nlyu/Code/maxrl-statistics/artifacts/bow-maxrl-word-het-example-r128/15-words_corr-0.2_len-128_pow-1.0_ar-0.5_hl-0.15
Dataset corr target: 0.2000
SNR halflife (word quantile): 0.15
Backbone lr: 1.230e-03
Head lr:     8.192e-03
Rollouts/sample: 128
Canonical degree: 128
Policy stdev:     1.0


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [3]:
state.run_training()

maxrl epoch 0:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 0:   0%|          | 0/390 [00:00<?, ?it/s]

maxrl epoch 1:   0%|          | 0/781 [00:00<?, ?it/s]

validation epoch 1:   0%|          | 0/390 [00:00<?, ?it/s]

## Results

Training saves:
1. A compact `metrics.parquet` to disk which contains per-epoch sufficient statistics to compute metrics.
2. Validation parquet containing per-row ground-truth and target.

In [4]:
metrics_path = config.study_folder / "metrics.parquet"
metrics = pl.read_parquet(metrics_path)
metrics

epoch,train_target_xx,train_target_xy,train_target_yy,train_target_n,train_ground_truth_xx,train_ground_truth_xy,train_ground_truth_yy,train_ground_truth_n,val_target_xx,val_target_xy,val_target_yy,val_target_n,val_ground_truth_xx,val_ground_truth_xy,val_ground_truth_yy,val_ground_truth_n
i64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,7317.108887,847.790527,49943.628906,49984.0,7317.108887,896.104492,2004.203857,49984.0,4777.761719,1440.653076,50076.695312,49920.0,4777.761719,1391.055176,2002.146606,49920.0
1,3623.439209,1698.101562,49943.875,49984.0,3623.439209,1556.091431,2003.652588,49984.0,6603.889648,2989.862305,50076.695312,49920.0,6603.889648,2862.797363,2002.146606,49920.0


In [ ]:
analysis = BagOfWordsAnalysisConfig.from_grouped({
    "example": [(0, config.study_folder)]
})

analysis.plot_vs_epoch([
    rsq_expr(split="train", y="ground_truth"),
    rsq_expr(split="val", y="ground_truth"),
    corr_expr(split="train", y="ground_truth"),
    corr_expr(split="val", y="ground_truth"),
])

In [6]:
last_epoch = int(metrics["epoch"].max())
validation_path = config.study_folder / str(last_epoch) / "validation.parquet"
validation_df = pl.read_parquet(validation_path)
validation_df.head()

model_preds,ground_truth,target
f64,f64,f64
-0.792969,-0.367188,-0.098633
0.124023,0.0859375,-0.933594
-0.291016,0.03125,0.691406
-0.120605,0.0,0.096191
-0.339844,0.015625,-0.429688
